In [ ]:
!pip install -q kaggle
!kaggle datasets download -d zalando-research/fashionmnist
!unzip fashionmnist.zip -d fashionmnist/
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import os
import kagglehub
from matplotlib import pyplot as plt

Dataset URL: https://www.kaggle.com/datasets/zalando-research/fashionmnist
License(s): other
100% 68.8M/68.8M [00:00<00:00, 96.6MB/s]

Archive:  fashionmnist.zip
  inflating: fashionmnist/fashion-mnist_test.csv  
  inflating: fashionmnist/fashion-mnist_train.csv  
  inflating: fashionmnist/t10k-images-idx3-ubyte  
  inflating: fashionmnist/t10k-labels-idx1-ubyte  
  inflating: fashionmnist/train-images-idx3-ubyte  
  inflating: fashionmnist/train-labels-idx1-ubyte  


In [ ]:
train_df = pd.read_csv('fashionmnist/fashion-mnist_train.csv')
test_df = pd.read_csv('fashionmnist/fashion-mnist_test.csv')

In [ ]:
def image_from_row(row, df=train_df):
    return 2*df.iloc[row, 1:].values.reshape(28,28,1)/255 -1

def images_from_df(indices, df=train_df):
    return 2*df.iloc[indices, 1:].values.reshape(len(indices), 28,28,1)/255 -1

In [ ]:
def residual_block(x, filters):
    shortcut = x

    if x.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters,
            1,
            padding="same"
        )(shortcut)

    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(filters, 3, padding="same")(x)

    x = layers.Add()([x, shortcut])
    x = layers.LeakyReLU(0.2)(x)

    return x

In [ ]:
from tensorflow.keras import models, layers
def criar_critico():
  inputs = layers.Input((28,28,1))
  x = layers.Conv2D(64, 3, padding="same")(inputs)
  x = residual_block(x, 64)
  x = residual_block(x, 128)
  x = residual_block(x, 256)
  x = layers.Flatten()(x)
  x = layers.Dense(512)(x)
  outputs = layers.Dense(1)(x)
  return models.Model(inputs, outputs)
criar_critico().summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 28, 28,    │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 28, 28,    │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 28, 28,    │          0 │ conv2d_1[0][0]    │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 28, 28,    │     36,928 │ leaky_re_lu[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 28, 28,    │          0 │ conv2d_2[0][0],   │
│                     │ 64)               │            │ conv2d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 28, 28,    │          0 │ add[0][0]         │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 28, 28,    │     73,856 │ leaky_re_lu_1[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_2       │ (None, 28, 28,    │          0 │ conv2d_4[0][0]    │
│ (LeakyReLU)         │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 28, 28,    │    147,584 │ leaky_re_lu_2[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 28, 28,    │      8,320 │ leaky_re_lu_1[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 28, 28,    │          0 │ conv2d_5[0][0],   │
│                     │ 128)              │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_3       │ (None, 28, 28,    │          0 │ add_1[0][0]       │
│ (LeakyReLU)         │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 28, 28,    │    295,168 │ leaky_re_lu_3[0]… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_4       │ (None, 28, 28,    │          0 │ conv2d_7[0][0]    │
│ (LeakyReLU)         │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 28, 28,    │    590,080 │ leaky_re_lu_4[0]… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 28, 28,    │     33,024 │ leaky_re_lu_3[0]

 Total params: 103,984,001 (396.67 MB)

 Trainable params: 103,984,001 (396.67 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
GENERATOR_LATENT_DIM = 512
def criar_gerador():
  inputs = layers.Input((GENERATOR_LATENT_DIM,))
  x = layers.Dense(512)(inputs)
  x = layers.Reshape((4,4,32))(x)
  x = residual_block(x, 128)
  x = layers.UpSampling2D((2,2))(x)
  x = layers.BatchNormalization()(x)
  x = layers.UpSampling2D((2,2))(x)
  x = residual_block(x, 64)
  x = layers.UpSampling2D((2,2))(x)
  x = layers.BatchNormalization()(x)
  x = residual_block(x, 32)
  x = residual_block(x, 16)
  outputs = layers.Conv2D(1, (5, 5), activation='tanh')(x)
  return  models.Model(inputs, outputs)
criar_gerador().summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 512)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 512)       │    262,656 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 4, 4, 32)  │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 4, 4, 128) │     36,992 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_6       │ (None, 4, 4, 128) │          0 │ conv2d_10[0][0]   │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 4, 4, 128) │    147,584 │ leaky_re_lu_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 4, 4, 128) │      4,224 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 4, 4, 128) │          0 │ conv2d_11[0][0],  │
│                     │                   │            │ conv2d_9[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_7       │ (None, 4, 4, 128) │          0 │ add_3[0][0]       │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 8, 8, 128) │          0 │ leaky_re_lu_7[0]… │
│ (UpSampling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 8, 8, 128) │        512 │ up_sampling2d[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_1     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (UpSampling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 16, 16,    │     73,792 │ up_sampling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_8       │ (None, 16, 16,    │          0 │ conv2d_13[0][0]   │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_14 (Conv2D)  │ (None, 16, 16,    │     36,928 │ leaky_re_lu_8[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 16, 16,    │      8,256 │ up_sampling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 16, 16,    │          0 │ conv2d_14[0][0],  │
│                     │ 64)               │            │ conv2d_12[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_9       │ (None, 16, 16,    │          0 │ add_4[0][0]       │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 608,865 (2.32 MB)

 Trainable params: 608,481 (2.32 MB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
train_images = train_df.iloc[:, 1:].values.astype("float32")
train_images = train_images.reshape(-1, 28, 28, 1)

train_images = train_images / 127.5 - 1

def images_from_array(indices):
    return train_images[indices]


In [10]:
import os
if os.path.exists("checkpoints.zip") and not os.path.exists("checkpoints"):
  !unzip -t /content/checkpoints.zip
elif not os.path.exists("checkpoints"):
  os.mkdir("checkpoints")

Archive:  /content/checkpoints.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/checkpoints.zip or
        /content/checkpoints.zip.zip, and cannot find /content/checkpoints.zip.ZIP, period.


In [ ]:
import math
from tqdm.notebook import tqdm
from keras.losses import BinaryCrossentropy
from keras.optimizers import Adam
import pickle
import time
if os.path.exists("checkpoints.zip") and not os.path.exists("checkpoints"):
  !unzip checkpoints.zip -d checkpoints
elif not os.path.exists("checkpoints"):
  os.mkdir("checkpoints")


batch_size = 128
epochs = 50
lambda_gp = 10
num_batches = int(math.ceil(train_df.shape[0]/batch_size))
n_critic = 5
save_interval = 128
dataset = tf.data.Dataset.from_tensor_slices(train_images)
dataset = dataset.shuffle(
    len(train_images)
).batch(
    batch_size
).prefetch(
    tf.data.AUTOTUNE
)
if os.path.exists('loss_history.pkl'):
  loss_history = pickle.load(open('loss_history.pkl', 'rb'))
else:
  loss_history = []
if os.path.exists('grads_history.pkl'):
  grads_history = pickle.load(open('grads_history.pkl', 'rb'))
else:
  grads_history = {}
if os.path.exists('gp_history.pkl'):
  gp_history = pickle.load(open('gp_history.pkl', 'rb'))
else:
  gp_history = []
if os.path.exists('norm_history.pkl'):
  norm_history = pickle.load(open('norm_history.pkl', 'rb'))
else:
  norm_history = []
if os.path.exists('diversity_history.pkl'):
  diversity_history = pickle.load(open('diversity_history.pkl', 'rb'))
else:
  diversity_history = []

if os.path.exists('loss_gen_history.pkl'):
  loss_gen_history = pickle.load(open('loss_gen_history.pkl', 'rb'))
else:
  loss_gen_history = []

with tf.device('/device:GPU:0'):
  gerador = criar_gerador()
  critico = criar_critico()
  optimizer_gerador = Adam(learning_rate=0.0001, beta_1=0.0, beta_2=0.9)
  optimizer_critico = Adam(learning_rate=0.0001, beta_1=0.0, beta_2=0.9)


  ckpt = tf.train.Checkpoint(
      gerador=gerador,
      critico=critico,
      optimizer_gerador=optimizer_gerador,
      optimizer_critico=optimizer_critico,
      epoch=tf.Variable(0),
      batch=tf.Variable(0)
  )

  manager = tf.train.CheckpointManager(
      ckpt,
      "./checkpoints",
      max_to_keep=5
  )
  print(manager.latest_checkpoint)
  ckpt.restore(manager.latest_checkpoint)

  initial_epoch = 0
  initial_batch = 0
  if manager.latest_checkpoint:
      print("Checkpoint carregado!")
      print("Época:", int(ckpt.epoch))
      print("Batch:", int(ckpt.batch))
      initial_epoch =  int(ckpt.epoch.numpy())
      initial_batch = int(ckpt.batch.numpy())
  else:
      print("Treinamento do zero.")
  for epoch in tqdm(
      range(initial_epoch, epochs),
      initial=initial_epoch,
      total=epochs,
      desc="epoch"
  ):
      start_batch = initial_batch if epoch == initial_epoch else 0

      for batch_index, imagens_reais in tqdm(
          enumerate(dataset),
          initial=start_batch,
          total=num_batches,
          desc="batch",
          leave=False
      ):
          if batch_index < start_batch:
              continue

          tamanho_batch_atual = imagens_reais.shape[0]
          for _ in range(n_critic):
            t0 = time.time()
            z = tf.random.normal((tamanho_batch_atual, GENERATOR_LATENT_DIM))
            t1 = time.time()
            print(f"generating z={t1-t0:.3f}s")
            with tf.GradientTape() as tape:
                imagens_geradas = gerador(z)
                t2 = time.time()
                print(f"gerador(z)={t2-t1:.3f}s")
                imagens_reais += tf.random.normal(
                    imagens_reais.shape,
                    stddev=0.05
                )
                imagens_reais = tf.clip_by_value(
                    imagens_reais,
                    -1.0,
                    1.0
                )
                t3 = time.time()
                print(f"ruído={t3-t2:.3f}s")
                y_pred_gerado = critico(imagens_geradas)
                y_pred_real = critico(imagens_reais)
                t4 = time.time()
                print(f"forward={t4-t3:.3f}s")
                epsilon = tf.random.uniform(
                    [tamanho_batch_atual,1,1,1]
                )
                x_hat = (
                    epsilon*imagens_reais
                    +
                    (1-epsilon)*imagens_geradas
                )
                with tf.GradientTape() as gp_tape:
                    gp_tape.watch(x_hat)

                    pred = critico(x_hat)
                grad = gp_tape.gradient(pred, x_hat)
                norm = tf.sqrt(
                    tf.reduce_sum(
                        tf.square(grad),
                        axis=[1,2,3]
                    )
                )
                gp = tf.reduce_mean(
                    (norm - 1.0) ** 2
                )
                loss_critico = tf.reduce_mean(y_pred_gerado) - tf.reduce_mean(y_pred_real) + lambda_gp*gp

                if batch_index % save_interval == 0:
                  norm_history.append(tf.reduce_mean(norm).numpy())
                  gp_history.append(gp.numpy())
                  loss_history.append(loss_critico.numpy())
                t5 = time.time()
                print(f"loss={t5-t4:.3f}s")
                # wasserstein = tf.reduce_mean(y_pred_gerado) - tf.reduce_mean(y_pred_real)
            t6 = time.time()
            grads_critico = tape.gradient(loss_critico, critico.trainable_variables)
            optimizer_critico.apply_gradients(
                zip(grads_critico, critico.trainable_variables)
            )
            t7 = time.time()
            print(f"apply gradients to critic={t7-t6:.3f}s")
          z = tf.random.normal((tamanho_batch_atual, GENERATOR_LATENT_DIM))
          with tf.GradientTape() as tape:
              t8 = time.time()
              imagens_geradas = gerador(z)
              y_pred_gerado = critico(imagens_geradas)
              loss_gerador = -tf.reduce_mean(y_pred_gerado)
              loss_gen_history.append(loss_gerador)
              t9 = time.time()
              print(f"forward gerador={t9-t8:.3f}s")

          t11 = time.time()
          grads_gerador = tape.gradient(loss_gerador, gerador.trainable_variables)
          optimizer_gerador.apply_gradients(
              zip(grads_gerador, gerador.trainable_variables)
          )
          t12 = time.time()
          print(f"aplicar grads gerador={t12-t11:.3f}s")
          if batch_index % save_interval == 0:
            z = tf.random.normal((256, GENERATOR_LATENT_DIM))
            imgs = gerador(z)
            diversidade = tf.reduce_mean(
                tf.math.reduce_std(imgs, axis=0)
            )
            diversity_history.append(diversidade)
            t13 = time.time()
            for i, g in enumerate(grads_gerador):
                if g is not None:
                    if i in grads_history.keys():
                        grads_history[i].append(tf.reduce_mean(tf.abs(g)).numpy())
                    else:
                        grads_history[i] = [tf.reduce_mean(tf.abs(g)).numpy()]
            t14 = time.time()
            print(f"salvar grads={t14-t13:.3f}s")
            if batch_index == num_batches - 1:
              ckpt.epoch.assign(epoch + 1)
              ckpt.batch.assign(0)
            else:
              ckpt.epoch.assign(epoch)
              ckpt.batch.assign(batch_index + 1)
            save_path = manager.save()
            print("Checkpoint salvo em:", save_path)

[]
None
Treinamento do zero.


epoch:   0%|          | 0/50 [00:00<?, ?it/s]

batch:   0%|          | 0/469 [00:00<?, ?it/s]

generating z=0.001s
gerador(z)=1.513s
ruído=0.006s
forward=1.766s
loss=2.388s
apply gradients to critic=1.289s
generating z=0.001s
gerador(z)=0.075s
ruído=0.001s
forward=0.053s
loss=0.243s
apply gradients to critic=0.573s
generating z=0.001s
gerador(z)=0.072s
ruído=0.001s
forward=0.039s
loss=0.259s
apply gradients to critic=0.569s
generating z=0.001s
gerador(z)=0.071s
ruído=0.001s
forward=0.037s
loss=0.262s
apply gradients to critic=0.583s
generating z=0.001s
gerador(z)=0.071s
ruído=0.001s
forward=0.038s
loss=0.261s
apply gradients to critic=0.572s
forward gerador=0.093s
aplicar grads gerador=0.443s
salvar grads=0.014s
Checkpoint salvo em: ./checkpoints/ckpt-1
generating z=0.001s
gerador(z)=0.049s
ruído=0.002s
forward=0.046s
loss=0.034s
apply gradients to critic=0.777s
generating z=0.001s
gerador(z)=0.074s
ruído=0.001s
forward=0.039s
loss=0.034s
apply gradients to critic=0.772s
generating z=0.001s
gerador(z)=0.071s
ruído=0.001s
forward=0.038s
loss=0.034s
apply gradients to critic=0.783

KeyboardInterrupt: 

**When the training needs to be stopped run the cells bellow to save the checkpoints and results on the repository**

In [ ]:
plt.figure()
plt.suptitle("Gradients")
for indice, valores in grads_history.items():
    plt.plot(valores)
plt.savefig('gradient_history.png')
plt.show()
plt.figure()
plt.suptitle("Losses")
plt.plot(loss_history, label='loss')
plt.savefig('loss_history.png')
plt.show()
plt.suptitle("GRADIENT PENALTY")
plt.plot(gp_history, label='gradient penalty')
plt.savefig('gp_history.png')
plt.show()
plt.suptitle("‖∇D(x̂)‖")
plt.plot(norm_history, label='norm')
plt.savefig('norm_history.png')
plt.show()
plt.suptitle("diversity")
plt.plot(diversity_history, label='diversity')
plt.savefig('diversity_history.png')
plt.show()
plt.suptitle("loss gerador")
plt.plot(loss_gen_history, label='loss_gerador')
plt.savefig('loss_gerador.png')
plt.show()

In [ ]:
import pickle
pickle.dump(grads_history, open('grads_history.pkl', 'wb'))
pickle.dump(loss_history, open('loss_history.pkl', 'wb'))
pickle.dump(gp_history, open('gp_history.pkl', 'wb'))
pickle.dump(norm_history, open('norm_history.pkl', 'wb'))
pickle.dump(diversity_history, open('diversity_history.pkl', 'wb'))
pickle.dump(loss_gen_history, open('loss_gen_history.pkl', 'wb'))

In [ ]:
os.makedirs('generator_outputs', exist_ok=True)
z = z = tf.random.normal((10, GENERATOR_LATENT_DIM))
imagens = gerador(z)
for i in range(10):
    plt.figure()
    plt.imsave(f'generator_outputs/{i}.png', tf.squeeze(imagens[i]), cmap="gray")

In [ ]:
!zip -r checkpoints.zip checkpoints
!zip -r generator_outputs.zip generator_outputs